# Module 3 - Class 1: Data Cleaning Lab

**Dataset:** Telco Customer Churn  
**Objective:** Learn to inspect, clean, and prepare raw data for modeling.

### What you will practice
- Inspecting data structure and types
- Detecting and fixing type conversion issues
- Handling missing values (mean, median, mode imputation)
- Checking for inconsistent categorical entries
- Outlier detection with IQR and Z-score methods

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")

## 1. Load Data

We load the Telco Customer Churn dataset. If the URL doesn't work, use the upload fallback cell below.

In [ ]:
# Primary: load from URL
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# Fallback: upload from local machine (uncomment if URL fails)
# from google.colab import files
# uploaded = files.upload()
# filename = list(uploaded.keys())[0]
# df = pd.read_csv(filename)
# print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")

## 2. Inspect Structure

In [ ]:
# Shape
print("Shape:", df.shape)
print()

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Statistical summary for numeric columns
df.describe()

In [ ]:
# Statistical summary for categorical columns
df.describe(include='object')

## 3. Fix the TotalCharges Type Issue

Notice that `TotalCharges` shows up as `object` (string) even though it should be numeric. Let's investigate and fix this.

In [ ]:
# Check the dtype
print("TotalCharges dtype:", df['TotalCharges'].dtype)

# Try converting - this will show us the problem rows
mask = pd.to_numeric(df['TotalCharges'], errors='coerce').isna()
print(f"\nRows that can't be converted to numeric: {mask.sum()}")
print("\nProblematic values:")
df.loc[mask, ['customerID', 'TotalCharges', 'tenure', 'MonthlyCharges']]

In [ ]:
# These are customers with tenure=0 (new customers) where TotalCharges is a blank string.
# Fix: convert to numeric, blanks become NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print("TotalCharges dtype after fix:", df['TotalCharges'].dtype)
print(f"NaN count in TotalCharges: {df['TotalCharges'].isna().sum()}")

In [ ]:
# For new customers (tenure=0), TotalCharges should logically be 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)
print(f"NaN count after fill: {df['TotalCharges'].isna().sum()}")

## 4. Missing Values

### 4a. Check for missing values across all columns

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'percent': missing_pct})
missing_df[missing_df['count'] > 0]

### 4b. Demonstrate imputation methods

We'll create artificial missing values in `MonthlyCharges` to practice different imputation strategies.

In [ ]:
# Create a copy and inject random NaNs for demonstration
np.random.seed(42)
df_demo = df.copy()
nan_idx = np.random.choice(df_demo.index, size=200, replace=False)
df_demo.loc[nan_idx, 'MonthlyCharges'] = np.nan
print(f"Injected {df_demo['MonthlyCharges'].isna().sum()} NaN values into MonthlyCharges")

In [ ]:
# Mean imputation
mean_val = df_demo['MonthlyCharges'].mean()
df_mean = df_demo.copy()
df_mean['MonthlyCharges'] = df_mean['MonthlyCharges'].fillna(mean_val)
print(f"Mean imputation value: {mean_val:.2f}")
print(f"Remaining NaN: {df_mean['MonthlyCharges'].isna().sum()}")

In [ ]:
# Median imputation
median_val = df_demo['MonthlyCharges'].median()
df_median = df_demo.copy()
df_median['MonthlyCharges'] = df_median['MonthlyCharges'].fillna(median_val)
print(f"Median imputation value: {median_val:.2f}")
print(f"Remaining NaN: {df_median['MonthlyCharges'].isna().sum()}")

### 4c. TODO: Mode Imputation + Missing Indicator

Complete the two tasks below.

In [ ]:
# TODO 1: Perform mode imputation on df_demo['MonthlyCharges']
# Hint: use df_demo['MonthlyCharges'].mode()[0]
# Store the result in df_mode

df_mode = df_demo.copy()
# YOUR CODE HERE


# TODO 2: Create a binary missing indicator column called 'MonthlyCharges_missing'
# It should be 1 where MonthlyCharges was NaN, 0 otherwise
# Add it to df_mode BEFORE filling NaN values

# YOUR CODE HERE


## 5. Inconsistent Entries

Check categorical columns for inconsistent or unexpected values.

In [ ]:
# Check unique values for key categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols.remove('customerID')  # skip ID

for col in cat_cols:
    vc = df[col].value_counts()
    print(f"--- {col} ({df[col].nunique()} unique) ---")
    print(vc)
    print()

In [ ]:
# Notice: some columns have 'No internet service' and 'No phone service' 
# which could be simplified to 'No' depending on your modeling choice.
# Let's check which columns have these:

for col in cat_cols:
    vals = df[col].unique()
    special = [v for v in vals if 'No ' in str(v) and v != 'No']
    if special:
        print(f"{col}: {special}")

In [ ]:
# Standardize: replace verbose 'No X service' with 'No'
replace_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in replace_cols:
    df[col] = df[col].replace('No internet service', 'No')

df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

print("Standardization complete.")
print("OnlineSecurity values:", df['OnlineSecurity'].unique())

## 6. Outlier Detection

### 6a. IQR Method (demonstrated)

In [ ]:
def detect_outliers_iqr(series, factor=1.5):
    """Detect outliers using the IQR method."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    outliers = (series < lower) | (series > upper)
    return outliers, lower, upper

# Apply to MonthlyCharges and TotalCharges
for col in ['MonthlyCharges', 'TotalCharges', 'tenure']:
    outliers, lower, upper = detect_outliers_iqr(df[col])
    print(f"{col}: {outliers.sum()} outliers (bounds: [{lower:.2f}, {upper:.2f}])")

In [ ]:
# Visualize distributions with box plots
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(['MonthlyCharges', 'TotalCharges', 'tenure']):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

### 6b. TODO: Z-score Method

Implement outlier detection using Z-scores. A data point is an outlier if its Z-score exceeds a threshold (typically 3).

In [ ]:
# TODO: Implement Z-score outlier detection
# 
# Z-score formula: z = (x - mean) / std
# A point is an outlier if |z| > threshold
#
# 1. Write a function detect_outliers_zscore(series, threshold=3)
#    that returns a boolean mask of outliers
# 2. Apply it to 'MonthlyCharges', 'TotalCharges', 'tenure'
# 3. Print how many outliers each column has
# 4. Compare results with IQR method above

def detect_outliers_zscore(series, threshold=3):
    # YOUR CODE HERE
    pass

# YOUR CODE HERE - apply and print results


## 7. Save Cleaned Data

In [ ]:
# Final check
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum().sum(), "total")
print("\nDtypes:")
print(df.dtypes.value_counts())

In [ ]:
# Save to CSV
df.to_csv('telco_churn_cleaned.csv', index=False)
print("Saved: telco_churn_cleaned.csv")

# Download in Colab
# from google.colab import files
# files.download('telco_churn_cleaned.csv')

---
## Summary

| Step | What we did |
|------|-------------|
| Inspect | `.info()`, `.describe()`, `.shape` |
| Type fix | `TotalCharges` string to numeric |
| Missing values | Mean/median imputation (TODO: mode + indicator) |
| Inconsistencies | Standardized 'No X service' values |
| Outliers | IQR method (TODO: Z-score) |